In [1]:
# connecting to the drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Preprocessing Structured Data - Selena


In [2]:
# adding path to CSVs

import csv
import pandas as pd
import numpy as np
from datetime import timedelta
import os

!ls '/content/drive/MyDrive/25 Fall/BME499 - Senior Capstone/499 Senior Project Group - RaGS/Code/DEMO CSV Files'


ADMISSIONS.csv			DIAGNOSES_ICD.csv     LABEVENTS.csv
all_features_table.csv		D_ICD_DIAGNOSES.csv   MICROBIOLOGYEVENTS.csv
all_features_table_encoded.csv	D_ICD_PROCEDURES.csv  OUTPUTEVENTS.csv
CALLOUT.csv			D_ITEMS.csv	      PATIENTS.csv
CAREGIVERS.csv			D_LABITEMS.csv	      PRESCRIPTIONS.csv
CHARTEVENTS.csv			DRGCODES.csv	      PROCEDUREEVENTS_MV.csv
CPTEVENTS.csv			ICUSTAYS.csv	      PROCEDURES_ICD.csv
DATETIMEEVENTS.csv		INPUTEVENTS_CV.csv    SERVICES.csv
D_CPT.csv			INPUTEVENTS_MV.csv    TRANSFERS.csv


In [3]:
# create vectors to store hadm-id's we want to use for training

def remove_duplicates(arr):
    """Remove duplicate values from an array while preserving order."""
    seen = set()
    unique = []
    for x in arr:
        if x not in seen:
            unique.append(x)
            seen.add(x)
    return unique


training_id = [142345,105331,165520,199207,177759,103770,199395,132349,140372,157235,110244,189483,111115,
               124073,126949,145203,174739,188574,167957,187023,157466,193924,182839,175237,172082];
# remove duplicates just in case
training_id = remove_duplicates(training_id)
print(training_id)


[142345, 105331, 165520, 199207, 177759, 103770, 199395, 132349, 140372, 157235, 110244, 189483, 111115, 124073, 126949, 145203, 174739, 188574, 167957, 187023, 157466, 193924, 182839, 175237, 172082]


In [4]:
# HELPER FUNCTIONS
# ---------------------------

def load_filtered_csv(filename, usecols=None, header=1):
    """Load CSV, keep only rows with hadm_id in training_id."""
    path = f"{datapath}/{filename}"

    dtype_overrides = {
        'subject_id': 'string',
        'hadm_id': 'string',
        'icustay_id': 'string',
        'itemid': 'string',
        'row_id': 'string',
        'dob': 'string'
    }

    df = pd.read_csv(path, usecols=usecols, header=1, dtype=dtype_overrides)
    df.columns = df.columns.str.strip().str.lower()
    if 'hadm_id' in df.columns:
        df = df[df['hadm_id'].isin([str(x) for x in training_id])]
    return df


def aggregate_in_bins(df, time_col, value_cols, ref_time_col='REF_TIME'):
    """Bin events into N_HOURS windows relative to ref_time_col."""
    results = []

    for hadm, sub in df.groupby('hadm_id'):
        ref_time = sub[ref_time_col].iloc[0] if ref_time_col in sub else sub[time_col].max()
        ref_time = pd.to_datetime(ref_time)
        sub[time_col] = pd.to_datetime(sub[time_col])

        # Compute time difference (hours) from reference
        sub['delta_h'] = (ref_time - sub[time_col]).dt.total_seconds() / 3600

        # Select only the last N_BINS * N_HOURS window
        sub = sub[sub['delta_h'] >= 0]
        sub = sub[sub['delta_h'] <= N_BINS * N_HOURS]

        # Bin index: 0 (oldest) to N_BINS-1 (most recent)
        sub['bin'] = (sub['delta_h'] // N_HOURS).astype(int)
        sub['bin'] = N_BINS - 1 - sub['bin']  # invert order (bin2 = most recent)

        # Aggregate by bin
        agg = sub.groupby('bin')[value_cols].agg(['mean','min','max','std'])
        agg.columns = ['_'.join([c, stat, f'bin{b}'])
                       for b in agg.index for c in value_cols for stat in ['mean','min','max','std']][:len(agg.columns)]
        agg['hadm_id'] = hadm
        results.append(agg.reset_index(drop=True))

    if results:
        return pd.concat(results, ignore_index=True)
    else:
        return pd.DataFrame()


# PARAMETERS
# ---------------------------

N_HOURS = 6          # length of each bin (hours)
N_BINS = 3           # number of temporal bins
AGG_FUN = 'mean'                # aggregation per bin



In [5]:
# loading in csv files

datapath = '/content/drive/MyDrive/25 Fall/BME499 - Senior Capstone/499 Senior Project Group - RaGS/Code/DEMO CSV Files/';

admissions = load_filtered_csv('ADMISSIONS.csv', usecols=['subject_id','hadm_id','admittime','admission_type','religion','marital_status','ethnicity'])
patients = load_filtered_csv('PATIENTS.csv', usecols=['subject_id','gender','dob'])

print(admissions.columns.tolist())

# for FUTURE:  add randomized times
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
patients['dob'] = pd.to_datetime(patients['dob'])
merged_static = admissions.merge(patients, on='subject_id', how='left')
merged_static['admittime'] = pd.to_datetime(merged_static['admittime'], errors='coerce')
merged_static['dob'] = pd.to_datetime(merged_static['dob'], errors='coerce')

# Compute age in days
#delta = (merged_static['admittime'] - merged_static['dob']).dt.total_seconds() / (3600 * 24)
#merged_static['age'] = delta / 365.25

# Cap maximum age to 90 (MIMIC policy)
# merged_static.loc[merged_static['age'] > 90, 'age'] = 90
static_features_table = merged_static[['hadm_id','gender','admission_type','ethnicity']]
# static_features = merged_static[['hadm_id','age','gender','admission_type','ethnicity']]

print(static_features_table)


['subject_id', 'hadm_id', 'admittime', 'admission_type', 'religion', 'marital_status', 'ethnicity']
   hadm_id gender admission_type                       ethnicity
0   142345      F      EMERGENCY          BLACK/AFRICAN AMERICAN
1   105331      F      EMERGENCY           UNKNOWN/NOT SPECIFIED
2   165520      F      EMERGENCY           UNKNOWN/NOT SPECIFIED
3   199207      F      EMERGENCY                           WHITE
4   177759      M      EMERGENCY                           WHITE
5   103770      F      EMERGENCY                           WHITE
6   199395      F       ELECTIVE                           WHITE
7   132349      M      EMERGENCY                           WHITE
8   140372      M      EMERGENCY                           WHITE
9   157235      F      EMERGENCY                           WHITE
10  110244      M       ELECTIVE                           WHITE
11  189483      F      EMERGENCY                           WHITE
12  111115      F      EMERGENCY                       

In [6]:
# Converting categorical bits of static_features into vectors





In [7]:
# creating reference times for each hadm_id
# ~~ currently just time of latest entry, but will want to randomize in the future

# Ensure datetime conversion
df = load_filtered_csv('CHARTEVENTS.csv', usecols=['hadm_id','charttime'])
hadm_col='hadm_id'
time_col='charttime'
#print(df)

df = df.copy()
df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
df = df.dropna(subset=[hadm_col, time_col])

# Group and get the latest time per admission
ref_times = (
    df.groupby(hadm_col, as_index=False)[time_col]
      .max()
      .rename(columns={time_col: 'ref_time'})
)

print(ref_times)

   hadm_id            ref_time
0   103770 2195-05-19 16:00:00
1   105331 2126-08-28 16:10:00
2   110244 2129-03-05 21:00:00
3   111115 2144-02-14 19:00:00
4   124073 2152-10-07 14:00:00
5   126949 2129-12-01 00:00:00
6   132349 2139-09-25 16:00:00
7   140372 2138-04-11 18:00:00
8   142345 2164-10-25 08:30:00
9   145203 2107-02-10 10:26:00
10  157235 2132-12-06 14:00:00
11  157466 2117-08-13 17:00:00
12  165520 2125-10-07 12:12:00
13  167957 2171-11-04 11:00:00
14  172082 2200-03-28 17:15:00
15  174739 2180-02-01 08:00:00
16  175237 2155-12-17 12:58:00
17  177759 2163-05-15 21:00:00
18  182839 2198-07-20 12:00:00
19  187023 2138-06-07 14:00:00
20  188574 2148-01-19 14:00:00
21  189483 2185-03-26 09:10:00
22  199207 2149-05-31 20:00:00
23  199395 2190-07-20 15:30:00


In [8]:
# finding the hadm-ids of similar items (i.e. "heart rate" and "heart rates")

import re

def find_itemids_by_label(df, keywords, case_insensitive=True, fuzzy=False):
    """
    Search D_ITEMS or D_LABITEMS for ITEMIDs whose LABEL or ABBREVIATION
    matches one or more given keywords.

    Parameters
    ----------
    dictionary_path : str
        Path to D_ITEMS.csv or D_LABITEMS.csv
    keywords : list of str
        Words/phrases to search for (e.g. ['heart rate', 'pulse'])
    case_insensitive : bool
        If True, match regardless of case.
    fuzzy : bool
        If True, use substring matching with word boundaries (slower but catches variations).

    Returns
    -------
    matches : pd.DataFrame
        Subset of the dictionary with matching ITEMIDs and LABELs
    """

    # Build regex pattern
    flags = re.IGNORECASE if case_insensitive else 0
    if fuzzy:
        pattern = '|'.join([fr'\b{re.escape(k)}\b' for k in keywords])
    else:
        pattern = '|'.join([re.escape(k) for k in keywords])

    # Combine label/abbreviation/description fields if they exist
    text_cols = [c for c in ['label', 'abbreviation', 'category', 'param_type', 'fluid', 'specimen'] if c in df.columns]
    df['search_text'] = df[text_cols].astype(str).agg(' '.join, axis=1)

    # Search
    mask = df['search_text'].str.contains(pattern, flags=flags, regex=True)
    matches = df[mask].copy()

    # Keep key columns
    keep_cols = [c for c in ['itemid','label','abbreviation','category','fluid','specimen'] if c in matches.columns]
    matches = matches[keep_cols].drop_duplicates(subset=['itemid'])

    # Sort alphabetically for readability
    return matches.sort_values('label').reset_index(drop=True)



In [9]:

# Time bin data helper functions

# loads in the temporal data sets
def load_temporal_table(filename, usecols, time_col, value_col, item_col=None, itemids=None, header=1):
    """
    Loads and prepares a temporal table (like CHARTEVENTS or LABEVENTS).
    Optionally maps ITEMIDs to meaningful labels.
    """
    path = f"{datapath}/{filename}"
    df = pd.read_csv(path, usecols=usecols, header=header, low_memory=False)
    df.columns = df.columns.str.strip().str.lower()

     # Filter by hadm_id
    df = df[df['hadm_id'].isin(training_id)]

    # Filter by itemids if provided
    if itemids is not None:
        # itemids might be a pandas Series or list; ensure it's a list
        itemid_list = list(itemids)
        df = df[df[item_col].isin(itemid_list)]

    # Convert to datetime
    df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
    df = df.dropna(subset=[time_col, value_col])

    return df


# can split to 3 timebins, find mean, max, min, and change between time bins
# will output NaN value does not exist
def bin_and_average(df, time_col, value_col, hadm_col='hadm_id', ref_time_col=None, AGG_FUN=('mean','max'), prefix=None, add_trend=True):
    """
    Aggregates a temporal dataframe into N_BINS × BIN_HOURS windows before ref_time_col.
    Returns per-hadm_id averaged values per bin.
    """
    results = []
    name_prefix = prefix if prefix else value_col

    # Pre-build output column names
    feature_cols = [hadm_col]
    for fun in AGG_FUN:
        for b in range(N_BINS):
            feature_cols.append(f'{name_prefix}_bin{b+1}_{fun}')
        if add_trend and fun == 'mean':
            for b in range(1, N_BINS):
                feature_cols.append(f'change_{name_prefix}_bin{b+1}_{fun}')

    # Empty or invalid DataFrame case
    if df is None or df.empty or not all(c in df.columns for c in [hadm_col, time_col, value_col]):
        print(f"⚠️ No valid data for {name_prefix}, returning NaN-filled feature table.")
        return pd.DataFrame(columns=feature_cols)

    # If ref_time_col not given, use last available time as reference
    if ref_time_col is None:
        df['ref_time'] = df.groupby(hadm_col)[time_col].transform('max')
        ref_time_col = 'ref_time'

    for hadm, sub in df.groupby(hadm_col):

        # Skip if subset empty or invalid
        if sub.empty or sub[value_col].dropna().empty:
            continue

        ref_time = pd.to_datetime(sub[ref_time_col].iloc[0])
        if pd.isna(ref_time):
            continue

        sub[time_col] = pd.to_datetime(sub[time_col])
        sub = sub.dropna(subset=[time_col, value_col])
        if sub.empty:
            continue

        # Calculate how many hours before reference
        sub['delta_h'] = (ref_time - sub[time_col]).dt.total_seconds() / 3600
        sub = sub[(sub['delta_h'] >= 0) & (sub['delta_h'] <= N_BINS * N_HOURS)]
        if sub.empty:
            # Fill with NaNs for all bins
            row = {hadm_col: hadm}
            name_prefix = prefix if prefix else value_col
            for fun in AGG_FUN:
                for b in range(N_BINS):
                    row[f'{name_prefix}_bin{b+1}_{fun}'] = np.nan
                if add_trend and fun == 'mean':
                    for b in range(1, N_BINS):
                        row[f'change_{name_prefix}_bin{b+1}_{fun}'] = np.nan
            results.append(row)
            continue

        # Compute bin index
        sub['bin'] = np.floor(sub['delta_h'] / N_HOURS).astype(int)
        sub['bin'] = N_BINS - 1 - sub['bin']  # bin2 = most recent

        row = {hadm_col: hadm}
        name_prefix = prefix if prefix else value_col

        # Aggregate per bin
        for fun in AGG_FUN:
            bin_stats = sub.groupby('bin')[value_col].agg(fun)
            for b in range(N_BINS):
                row[f'{prefix}_bin{b+1}_{fun}'] = bin_stats.get(b, np.nan)

            # add trend only for mean
            if add_trend and fun == 'mean':
                for b in range(1, N_BINS):
                    prev = bin_stats.get(b-1, np.nan)
                    curr = bin_stats.get(b, np.nan)
                    delta = curr - prev if pd.notna(curr) and pd.notna(prev) else np.nan
                    row[f'change_{prefix}_bin{b+1}_{fun}'] = delta

        results.append(row)

    # If all groups were empty
    if not results:
        print(f"⚠️ No valid data found for {prefix or value_col}. Returning NaN-filled table.")
        return pd.DataFrame(columns=feature_cols)
    else:
      df_out = pd.DataFrame(results)
      df_out = df_out.groupby('hadm_id').mean(numeric_only=True).reset_index()
      return df_out



In [10]:
# generating arrays of ids for vitals
d_items = load_filtered_csv('D_ITEMS.csv', usecols=['itemid','label'])

heart_ids = find_itemids_by_label(d_items, keywords=['heart rate','atrial rate'], fuzzy=True)
# print(heart_ids)
heart_ids1 = [211,220045];
# print(heart_ids)

sbp_ids = find_itemids_by_label(d_items, ['blood pressure','systolic'])
# print(sbp_ids)
sbp_ids1 = [6,225309,6701,51,220052,3313,3315,3317,3321,3323,480,482]

dbp_ids = find_itemids_by_label(d_items, ['blood pressure','diastolic'])
# print(dbp_ids)
dbp_ids1 = [8364,225310,8555,8368,220051,8502,8503,8504,8505,8506,8507,8444,8445,8446]

temp_ids = find_itemids_by_label(d_items, ['temperature','temp'])
# print(temp_ids)
temp_ids1 = [676,677,223762]

spo2_ids = find_itemids_by_label(d_items, ['spo2','oxygen saturation'])
# print(spo2_ids)
spo2_ids1 = [228232,646,6719]

MAP_ids = find_itemids_by_label(d_items, ['map','mean arterial pressure'])
# print(MAP_ids)
MAP_ids1 = [6399, 2309, 2544, 2974, 3067, 6605, 1199, 2522, 6579,438,2369,1321,2770,5804]


# FOR FUTURE - partially manually done, so may not emcompass all the applicable itemid's of a larger dataset... need to find a better system


In [11]:
# putting together temporal bins for vitals

# PULSE
vitals = load_temporal_table(
    filename='CHARTEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=heart_ids1
)

# (c) average across all heart-rate-type ITEMIDs per time
vitals_grouped = vitals.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()

# (d) bin into temporal windows
heart_features = bin_and_average(vitals_grouped, time_col='charttime', value_col='valuenum', prefix="pulse")

print(heart_features.head())


# SYS BLOOD PRESSURE
vitals = load_temporal_table(
    filename='CHARTEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=sbp_ids1
)

# (c) get rid of duplicate entries
vitals_grouped = vitals.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()

# (d) bin into temporal windows
sys_bp_features = bin_and_average(vitals_grouped, time_col='charttime', value_col='valuenum', prefix="sys_bp")

print(sys_bp_features.head())


# DIAS BLOOD PRESSURE
vitals = load_temporal_table(
    filename='CHARTEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=dbp_ids1
)

# (c)  get rid of duplicate entries
vitals_grouped = vitals.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()

# (d) bin into temporal windows
dias_bp_features = bin_and_average(vitals_grouped, time_col='charttime', value_col='valuenum', prefix="dias_bp")

print(dias_bp_features.head())


# MAP
vitals = load_temporal_table(
    filename='CHARTEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=MAP_ids1
)

# (c)  get rid of duplicate entries
vitals_grouped = vitals.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
print(vitals_grouped)

# (d) bin into temporal windows
MAP_features = bin_and_average(vitals_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean','min'), prefix="MAP")

print(MAP_features.head())


# TEMP
vitals = load_temporal_table(
    filename='CHARTEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=temp_ids1
)

# (c) get rid of duplicate entries
vitals_grouped = vitals.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()

# (d) bin into temporal windows
temp_features = bin_and_average(vitals_grouped, time_col='charttime', value_col='valuenum', prefix="temp")

print(temp_features.head())


# SPO2
vitals = load_temporal_table(
    filename='CHARTEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=spo2_ids1
)

# (c) get rid of duplicate entries
vitals_grouped = vitals.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()

# (d) bin into temporal windows
spo2_features = bin_and_average(vitals_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean','min'), prefix="spo2")

print(spo2_features.head())




   hadm_id  pulse_bin1_mean  pulse_bin2_mean  pulse_bin3_mean  \
0   103770        65.428571        62.000000        63.000000   
1   105331       117.666667       136.833333       141.375000   
2   110244        66.166667        73.833333        76.666667   
3   111115        85.000000        86.666667        82.000000   
4   124073        64.571429        65.400000        69.142857   

   change_pulse_bin2_mean  change_pulse_bin3_mean  pulse_bin1_max  \
0               -3.428571                1.000000            71.0   
1               19.166667                4.541667           136.0   
2                7.666667                2.833333            73.0   
3                1.666667               -4.666667            88.0   
4                0.828571                3.742857            67.0   

   pulse_bin2_max  pulse_bin3_max  
0            68.0            69.0  
1           154.0           158.0  
2            87.0            87.0  
3            91.0            83.0  
4            7

In [12]:
# combining all vitals feature tables into a single one

from functools import reduce

vitals_features = [heart_features, sys_bp_features, dias_bp_features, MAP_features, temp_features, spo2_features]

vitals_feature_table = reduce(
    lambda left, right: pd.merge(left, right, on='hadm_id', how='outer'),  vitals_features)

# print(vitals_feature_table)
list(vitals_feature_table.columns)

['hadm_id',
 'pulse_bin1_mean',
 'pulse_bin2_mean',
 'pulse_bin3_mean',
 'change_pulse_bin2_mean',
 'change_pulse_bin3_mean',
 'pulse_bin1_max',
 'pulse_bin2_max',
 'pulse_bin3_max',
 'sys_bp_bin1_mean',
 'sys_bp_bin2_mean',
 'sys_bp_bin3_mean',
 'change_sys_bp_bin2_mean',
 'change_sys_bp_bin3_mean',
 'sys_bp_bin1_max',
 'sys_bp_bin2_max',
 'sys_bp_bin3_max',
 'dias_bp_bin1_mean',
 'dias_bp_bin2_mean',
 'dias_bp_bin3_mean',
 'change_dias_bp_bin2_mean',
 'change_dias_bp_bin3_mean',
 'dias_bp_bin1_max',
 'dias_bp_bin2_max',
 'dias_bp_bin3_max',
 'MAP_bin1_mean',
 'MAP_bin2_mean',
 'MAP_bin3_mean',
 'change_MAP_bin2_mean',
 'change_MAP_bin3_mean',
 'MAP_bin1_min',
 'MAP_bin2_min',
 'MAP_bin3_min',
 'temp_bin1_mean',
 'temp_bin2_mean',
 'temp_bin3_mean',
 'change_temp_bin2_mean',
 'change_temp_bin3_mean',
 'temp_bin1_max',
 'temp_bin2_max',
 'temp_bin3_max',
 'spo2_bin1_mean',
 'spo2_bin2_mean',
 'spo2_bin3_mean',
 'change_spo2_bin2_mean',
 'change_spo2_bin3_mean',
 'spo2_bin1_min',
 'spo2

In [13]:
# calculating MAP if empty, and if blood pressure is given

# vitals_feature_table1 = vitals_feature_table.copy()

# timebin 1
map_cols = [c for c in vitals_feature_table.columns if "MAP_bin1_mean" in c]
sbp_cols = [c for c in vitals_feature_table.columns if "sys_bp_bin1_mean" in c]
dbp_cols = [c for c in vitals_feature_table.columns if "dias_bp_bin1_mean" in c]

print(map_cols, sbp_cols, dbp_cols)

for map_col, sbp_col, dbp_col in zip(map_cols, sbp_cols, dbp_cols):
    print(map_col, sbp_col, dbp_col)

    mask = vitals_feature_table[map_col].isna() & \
           vitals_feature_table[sbp_col].notna() & \
           vitals_feature_table[dbp_col].notna()

    vitals_feature_table.loc[mask, map_col] = (
        vitals_feature_table.loc[mask, sbp_col] +
        2 * vitals_feature_table.loc[mask, dbp_col]
    ) / 3


# time bin2

map_cols = [c for c in vitals_feature_table.columns if "MAP_bin2_mean" in c]
sbp_cols = [c for c in vitals_feature_table.columns if "sys_bp_bin2_mean" in c]
dbp_cols = [c for c in vitals_feature_table.columns if "dias_bp_bin2_mean" in c]

for map_col, sbp_col, dbp_col in zip(map_cols, sbp_cols, dbp_cols):

    mask = vitals_feature_table[map_col].isna() & \
           vitals_feature_table[sbp_col].notna() & \
           vitals_feature_table[dbp_col].notna()

    vitals_feature_table.loc[mask, map_col] = (
        vitals_feature_table.loc[mask, sbp_col] +
        2 * vitals_feature_table.loc[mask, dbp_col]
    ) / 3


# time bin 3

map_cols = [c for c in vitals_feature_table.columns if "MAP_bin3_mean" in c]
sbp_cols = [c for c in vitals_feature_table.columns if "sys_bp_bin3_mean" in c]
dbp_cols = [c for c in vitals_feature_table.columns if "dias_bp_bin3_mean" in c]

for map_col, sbp_col, dbp_col in zip(map_cols, sbp_cols, dbp_cols):

    mask = vitals_feature_table[map_col].isna() & \
           vitals_feature_table[sbp_col].notna() & \
           vitals_feature_table[dbp_col].notna()

    vitals_feature_table.loc[mask, map_col] = (
        vitals_feature_table.loc[mask, sbp_col] +
        2 * vitals_feature_table.loc[mask, dbp_col]
    ) / 3


# change in MAP

map1_cols = [c for c in vitals_feature_table.columns if "MAP_bin1_mean" in c]
map2_cols = [c for c in vitals_feature_table.columns if "MAP_bin2_mean" in c]
map3_cols = [c for c in vitals_feature_table.columns if "MAP_bin3_mean" in c]

dmap2_cols = [c for c in vitals_feature_table.columns if "change_MAP_bin2_mean" in c]
dmap3_cols = [c for c in vitals_feature_table.columns if "change_MAP_bin3_mean" in c]

for map1_col, map2_col, map3_col, dmap2_col, dmap3_col in zip(map1_cols, map2_cols, map3_cols, dmap2_cols, dmap3_cols):

    # change between time bins 1 and 2
    mask2 = vitals_feature_table[dmap2_col].isna() & \
           vitals_feature_table[map1_col].notna() & \
           vitals_feature_table[map2_col].notna()

    # change between time bins 2 and 3
    mask3 = vitals_feature_table[dmap3_col].isna() & \
           vitals_feature_table[map2_col].notna() & \
           vitals_feature_table[map3_col].notna()

    vitals_feature_table.loc[mask2, dmap2_col] = (
        vitals_feature_table.loc[mask, map2_col] -
        vitals_feature_table.loc[mask, map1_col])

    vitals_feature_table.loc[mask3, dmap3_col] = (
        vitals_feature_table.loc[mask, map3_col] -
        vitals_feature_table.loc[mask, map2_col])


# print(vitals_feature_table['MAP_bin1_mean'])
list(vitals_feature_table.columns)

['MAP_bin1_mean'] ['sys_bp_bin1_mean'] ['dias_bp_bin1_mean']
MAP_bin1_mean sys_bp_bin1_mean dias_bp_bin1_mean


['hadm_id',
 'pulse_bin1_mean',
 'pulse_bin2_mean',
 'pulse_bin3_mean',
 'change_pulse_bin2_mean',
 'change_pulse_bin3_mean',
 'pulse_bin1_max',
 'pulse_bin2_max',
 'pulse_bin3_max',
 'sys_bp_bin1_mean',
 'sys_bp_bin2_mean',
 'sys_bp_bin3_mean',
 'change_sys_bp_bin2_mean',
 'change_sys_bp_bin3_mean',
 'sys_bp_bin1_max',
 'sys_bp_bin2_max',
 'sys_bp_bin3_max',
 'dias_bp_bin1_mean',
 'dias_bp_bin2_mean',
 'dias_bp_bin3_mean',
 'change_dias_bp_bin2_mean',
 'change_dias_bp_bin3_mean',
 'dias_bp_bin1_max',
 'dias_bp_bin2_max',
 'dias_bp_bin3_max',
 'MAP_bin1_mean',
 'MAP_bin2_mean',
 'MAP_bin3_mean',
 'change_MAP_bin2_mean',
 'change_MAP_bin3_mean',
 'MAP_bin1_min',
 'MAP_bin2_min',
 'MAP_bin3_min',
 'temp_bin1_mean',
 'temp_bin2_mean',
 'temp_bin3_mean',
 'change_temp_bin2_mean',
 'change_temp_bin3_mean',
 'temp_bin1_max',
 'temp_bin2_max',
 'temp_bin3_max',
 'spo2_bin1_mean',
 'spo2_bin2_mean',
 'spo2_bin3_mean',
 'change_spo2_bin2_mean',
 'change_spo2_bin3_mean',
 'spo2_bin1_min',
 'spo2

In [14]:
# finding itemids for lab

d_lab = load_filtered_csv('D_LABITEMS.csv', usecols=['itemid','label','fluid','category'])

# CREATININE - urine
lab_keywords = ['creatinine']
creatinine_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
#print(creatinine_ids)
creatinine_ids1 = [51067,51080,51081,51006]

# WBC - urine + blood
lab_keywords = ['wbc','white blood']
wbc_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
#print(wbc_ids)
wbc_urine_ids1 = [51516,51517,51518]
wbc_blood_ids1 = [51300]

# LACTATE
lab_keywords = ['lactate']
lactate_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
#print(lactate_ids)
lactate_ids1 = [50813]

# BUN - blood urea nitrogran
lab_keywords = ['nitrogen','urea','bun']
BUN_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
#print(BUN_ids)
BUN_ids1 = [51006]

# AST - aspartate aminotransferase
lab_keywords = ['aspartate','aminotransferase', 'ast']
AST_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
#print(AST_ids)
AST_ids1 = [50878]

# ALT - alanine aminotransferase
ALT_ids1 = [50861]

# BILIRUBIN - urine + blood direct / total
lab_keywords = ['bilirubin']
bilirubin_urine_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
#print(bilirubin_ids)
bilirubin_urine_ids1 = [51464,51465]
bilirubin_dir_ids1 = [50883]
bilirubin_tot_ids1 = [50885]

# GLUCOSE - urine + blood
lab_keywords = ['glucose']
gluc_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
#print(gluc_ids)
gluc_blood_ids1 = [50809,50931,51529]
gluc_urine_ids1 = [51478,51084]

# Na - blood + urine
lab_keywords = ['na','sodium']
Na_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
#print(Na_ids)
Na_blood_ids1 = [50983,50824]
Na_urine_ids1 = [51100]

# K - blood + urine
lab_keywords = ['k','potassium']
pott_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
#print(pott_ids)
pott_blood_ids1 = [50971,50822]
pott_urine_ids1 = [51097]

# BICARBONATE - blood + urine
lab_keywords = ['bicarbonate','bicarb']
bicarb_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
#print(bicarb_ids)
bicarb_blood_ids1 = [50882,50803]
bicarb_urine_ids1 = [51076]

# HEMOGLOBIN
lab_keywords = ['hemoglobin']
hemoglob_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
#print(hemoglob_ids)
hemoglob_ids1 = [50811,51222]

# PLATELETS
lab_keywords = ['platelets']
platelets_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
#print(platelets_ids)
platelets_ids1 = [51240]

# blood pH
lab_keywords = ['pH']
blood_pH_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
# print(blood_pH_ids)
blood_pH_ids1 = [50820]

# blood CO2
lab_keywords = ['pCO2']
blood_pCO2_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
#print(blood_pCO2_ids)
blood_pCO2_ids1 = [50818]

# blood O2
lab_keywords = ['pO2']
blood_pO2_ids = find_itemids_by_label(d_lab, lab_keywords, fuzzy=True)
#print(blood_pO2_ids)
blood_pO2_ids1 = [50821]


In [15]:
# turning lab stuff into temporal bins

# WBC urine
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','itemid','charttime','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=wbc_urine_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
WBC_urine_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="WBC_urine",)
#print(WBC_urine_features.head())

# WBC blood
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=wbc_blood_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
WBC_blood_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="WBC_blood",)
#print(WBC_blood_features.head())

# Lactate
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=lactate_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
lactate_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="lactate",)
#print(lactate_features.head())

# Creatinine
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=creatinine_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
creatinine_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="creatinine",)
#print(creatinine_features.head())

# BUN
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=BUN_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
BUN_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="BUN",)
#print(BUN_features.head())

# AST
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=AST_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
AST_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="AST",)
#print(AST_features.head())

# ALT
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=ALT_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
ALT_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="ALT",)
#print(ALT_features.head())

# BILIRUBIN - urine / dir / tot
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=bilirubin_urine_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
bili_urine_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="bili_urine",)
#print(bili_urine_features.head())

labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=bilirubin_dir_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
bili_dir_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="bili_dir",)
#print(bili_dir_features.head())

labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=bilirubin_tot_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
bili_tot_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="bili_tot",)
#print(bili_tot_features.head())

# GLUCOSE - urine / blood
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=gluc_urine_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
gluc_urine_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="gluc_urine",)
#print(gluc_urine_features.head())

labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=gluc_blood_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
gluc_blood_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="gluc_blood",)
#print(gluc_blood_features.head())

# Na - urine / blood
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=Na_urine_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
Na_urine_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="Na_urine",)
#print(Na_urine_features.head())

labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=Na_blood_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
Na_blood_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="Na_blood",)
#print(Na_blood_features.head())

# K - urine / blood
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=pott_urine_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
pott_urine_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="pott_urine",)
#print(pott_urine_features.head())

labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=pott_blood_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
pott_blood_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="pott_blood",)
#print(pott_blood_features.head())

# BICARBONATE - urine/blood
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=bicarb_urine_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
bicarb_urine_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="bicarb_urine",)
#print(bicarb_urine_features.head())

labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=bicarb_blood_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
bicarb_blood_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="bicarb_blood",)
#print(bicarb_blood_features.head())

# HEMOGLOBIN
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=hemoglob_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
hemoglob_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="hemoglob",)
#print(hemoglob_features.head())

# PLATELETS
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=platelets_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
platelets_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="platelets",)
#print(platelets_features.head())

# pH, pCO2, pO2
labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=blood_pH_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
blood_pH_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="blood_pH",)
#print(blood_pH_features.head())

labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=blood_pCO2_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
blood_pCO2_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="blood_pCO2",)
#print(blood_pCO2_features.head())

labs = load_temporal_table(
    filename='LABEVENTS.csv',
    usecols=['hadm_id','charttime','itemid','valuenum'],
    time_col='charttime',
    value_col='valuenum',
    item_col='itemid',
    itemids=blood_pO2_ids1
)
labs_grouped = labs.groupby(['hadm_id','charttime'])['valuenum'].mean().reset_index()
blood_pO2_features = bin_and_average(labs_grouped, time_col='charttime', value_col='valuenum', AGG_FUN=('mean',), prefix="blood_pO2",)
#print(blood_pO2_features.head())


⚠️ No valid data for WBC_blood, returning NaN-filled feature table.
⚠️ No valid data for bili_urine, returning NaN-filled feature table.
⚠️ No valid data for bicarb_urine, returning NaN-filled feature table.
⚠️ No valid data for platelets, returning NaN-filled feature table.


In [16]:
# combining the lab stuff into 1 feature table

labs_features = [WBC_urine_features,WBC_blood_features,lactate_features,creatinine_features,BUN_features,
                 AST_features,ALT_features,bili_urine_features,bili_dir_features,bili_tot_features,
                 gluc_urine_features,gluc_blood_features,Na_urine_features,Na_blood_features,pott_urine_features,
                 pott_blood_features,bicarb_urine_features,bicarb_blood_features,hemoglob_features,
                 platelets_features,blood_pH_features,blood_pCO2_features,blood_pO2_features,]

labs_feature_table = reduce(
    lambda left, right: pd.merge(left, right, on='hadm_id', how='outer'),  labs_features)

print(labs_feature_table)




     hadm_id  WBC_urine_bin1_mean  WBC_urine_bin2_mean  WBC_urine_bin3_mean  \
0   103770.0                  NaN                  NaN                161.0   
1   105331.0                  NaN                  NaN                  2.0   
2   110244.0                  NaN                  NaN                  0.0   
3   111115.0                  NaN                  NaN                  NaN   
4   124073.0                  NaN                  NaN                  NaN   
5   126949.0                  NaN                  NaN                 10.0   
6   132349.0                  NaN                  NaN                  3.0   
7   140372.0                  NaN                  NaN                  NaN   
8   142345.0                  NaN                  NaN                  NaN   
9   145203.0                  NaN                  NaN                  8.0   
10  157235.0                  NaN                  NaN                  NaN   
11  157466.0                  NaN                  N

In [17]:
# Microbiology IDs
# ~~ no need to do bins ~~

micro = pd.read_csv(
    f"{datapath}/MICROBIOLOGYEVENTS.csv",
    usecols=['hadm_id','spec_type_desc','org_name','ab_name','interpretation',],
    header=1,low_memory=False
)
micro = micro[micro['hadm_id'].isin(training_id)]
micro['interpretation'] = micro['interpretation'].str.strip().str.upper()

# Keep valid interpretations (S, I, R) / removing NULLS
micro = micro[micro['interpretation'].isin(['S', 'I', 'R'])]

# Flag resistant (R) and intermediate (I)
micro['is_resistant'] = (micro['interpretation'] == 'R').astype(int)
micro['is_intermediate'] = (micro['interpretation'] == 'I').astype(int)

# If microorganisms found
culture_presence = (
    micro.groupby('hadm_id')['org_name'].nunique().reset_index(name='num_organisms')
)
culture_presence['any_pos_culture'] = (culture_presence['num_organisms'] > 0).astype(int)

# Proportion of resistant antibiotics among all tested
resistance_summary = (
    micro.groupby('hadm_id')['is_resistant'].mean().reset_index(name='prop_resist')
)

# Count of total antibiotics tested
num_tests = (
    micro.groupby('hadm_id')['interpretation'].count().reset_index(name='num_antibiotics')
)

# Mean proportion of intermediate results
intermediate_summary = (
    micro.groupby('hadm_id')['is_intermediate'].mean().reset_index(name='prop_intermed')
)

# creating feature table
micro_features_table = (
    culture_presence
    .merge(resistance_summary, on='hadm_id', how='outer')
    .merge(intermediate_summary, on='hadm_id', how='outer')
    .merge(num_tests, on='hadm_id', how='outer')
)

# fill NaNs for patients with no microbiology data
micro_features_table[['num_organisms','any_pos_culture','prop_resist','prop_intermed','num_antibiotics']] = \
    micro_features_table[['num_organisms','any_pos_culture','prop_resist','prop_intermed','num_antibiotics']].fillna(0)

print(micro_features_table)


    hadm_id  num_organisms  any_pos_culture  prop_resist  prop_intermed  \
0    103770              2                1     0.078947       0.000000   
1    111115              1                1     0.200000       0.000000   
2    132349              4                1     0.236842       0.052632   
3    142345              1                1     0.428571       0.142857   
4    145203              1                1     0.133333       0.000000   
5    157466              1                1     0.037037       0.000000   
6    172082              1                1     0.428571       0.000000   
7    175237              1                1     0.214286       0.000000   
8    177759              3                1     0.100000       0.000000   
9    182839              2                1     0.111111       0.000000   
10   187023              1                1     0.125000       0.000000   
11   193924              1                1     0.000000       0.000000   

    num_antibiotics  
0 

In [18]:
# helper function for finding overall across 3 time bins

def summarize_over_window(df,ref_times,hadm_col='hadm_id',time_col='charttime',value_col='valuenum',
    BIN_HOURS=N_HOURS, N_BIN=N_BINS, AGG_FUN=('mean', 'max'), prefix='NONE',
fill_missing=True
):
    """
    Summarize time-series data over the total time span of N_BINS × BIN_HOURS hours
    before each reference time (ref_time) for each admission.

    Parameters
    ----------
    df : pd.DataFrame
        Must include hadm_id, time_col, and value_col.
    ref_times : pd.DataFrame
        Must include hadm_id and ref_time (e.g., last charttime or admittime).
    hadm_col, time_col, value_col : str
        Column names for admission ID, timestamp, and numeric value.
    BIN_HOURS, N_BINS : int
        The number of bins and their width — defines total window length.
    agg_funs : tuple
        Aggregation functions (e.g., ('mean','sum','max','min')).
    prefix : str
        Prefix for output feature column names.
    fill_missing : bool
        If True, returns zero/NaN-filled rows for missing hadm_ids.

    Returns
    -------
    pd.DataFrame
        One row per hadm_id with aggregated features.
    """

    df = df.copy()
    ref_times = ref_times.copy()

    # Normalize column names
    df.columns = df.columns.str.strip().str.lower()
    ref_times.columns = ref_times.columns.str.strip().str.lower()
    hadm_col = hadm_col.lower()
    time_col = time_col.lower()
    value_col = value_col.lower()

    # Safety checks
    for col in [hadm_col, time_col, value_col]:
        if col not in df.columns:
            raise KeyError(f"Column '{col}' not found in df: {df.columns.tolist()}")
    if hadm_col not in ref_times.columns:
        raise KeyError(f"Column '{hadm_col}' not found in ref_times: {ref_times.columns.tolist()}")

    # Ensure datetimes
    df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
    ref_times['ref_time'] = pd.to_datetime(ref_times['ref_time'], errors='coerce')

    # Merge reference times
    merged = df.merge(ref_times[[hadm_col, 'ref_time']], on=hadm_col, how='right')

    # Calculate time difference
    merged['delta'] = (merged['ref_time'] - merged[time_col]).dt.total_seconds() / 3600.0

    # Filter valid time window
    merged = merged[(merged['delta'] >= 0) & (merged['delta'] <= BIN_HOURS * N_BIN)]

    if merged.empty:
        print("⚠️ No matching rows found for time window; returning empty feature table.")
        out = ref_times[[hadm_col]].copy()
        for fun in AGG_FUN:
            out[f'{prefix}_{fun}'] = np.nan
        return out

    # Group and aggregate
    grouped = merged.groupby(hadm_col)[value_col].agg(AGG_FUN).reset_index()

    # Rename output columns
    grouped.columns = [hadm_col] + [f"{prefix}_{f}" for f in AGG_FUN]

    # Optionally include all hadm_ids (even those without data)
    if fill_missing:
        grouped = ref_times[[hadm_col]].merge(grouped, on=hadm_col, how='left').fillna(0)

    return grouped




In [19]:
# getting inputevent itemids that correspond to fluid intake

d_items = pd.read_csv(f"{datapath}/D_ITEMS.csv", header=1, low_memory=False)
d_items.columns = d_items.columns.str.strip().str.lower()

# Filter for fluids in inputevents tables
fluids = d_items[
    ((d_items['category'] == 'Fluids/Intake') | (d_items['category'] == 'Fluids - Other (Not In Use)')) &
    (d_items['linksto'].str.contains('inputevents', case=False, na=False))
].copy()

# Sort and clean
fluids_ids = fluids[['itemid', 'label', 'category', 'linksto']].sort_values('label')
#print(fluids_ids)



In [20]:
# Making input tables across all time bins

# loading both INPUTEVENTS_MV and INPUTEVENTS_CV - just looking at IV fluid inputs

cols_mv = ['hadm_id', 'endtime', 'itemid', 'amount', 'amountuom']
input_mv = pd.read_csv(
    f"{datapath}/INPUTEVENTS_MV.csv", usecols=cols_mv, header=1,low_memory=False)
  # endtime will be used as charttime
if 'endtime' in input_mv.columns:
    input_mv.rename(columns={'endtime': 'charttime'}, inplace=True)
#print(input_mv)

cols_cv = ['hadm_id', 'charttime', 'itemid', 'amount', 'amountuom']
input_cv = pd.read_csv(
    f"{datapath}/INPUTEVENTS_CV.csv", usecols=cols_cv, header=1,low_memory=False)
#print(input_cv)


# Normalize column names
input_mv.columns = input_mv.columns.str.strip().str.lower()
input_cv.columns = input_cv.columns.str.strip().str.lower()

# Ensure consistent dtypes
for df in [input_mv, input_cv]:
    df['hadm_id'] = pd.to_numeric(df['hadm_id'], errors='coerce')
    df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
#    df['charttime'] = pd.to_datetime(df['charttime'], errors='coerce')
#    df.dropna(subset=['hadm_id', 'charttime', 'amount'], inplace=True)
    df['hadm_id'] = df['hadm_id'].astype(int)

# Combine them
input_combined = pd.concat([input_mv, input_cv], axis=0, ignore_index=True)
input_combined_training = input_combined[input_combined['hadm_id'].isin(training_id)]
#print(input_combined_training)


#print(fluids_ids)
input_combined_training = input_combined_training[input_combined_training['itemid'].isin(fluids_ids['itemid'])]
#print(input_combined_training)

#print(ref_times)

input_combined_training.columns = input_combined_training.columns.str.strip().str.lower()
ref_times.columns = ref_times.columns.str.strip().str.lower()
ref_times['hadm_id'] = pd.to_numeric(ref_times['hadm_id'], errors='coerce').astype('Int64')


# Extract overall timebins feature
input_features_table = summarize_over_window(
    df=input_combined_training,
    ref_times=ref_times,
    hadm_col='hadm_id',
    time_col='charttime',
    value_col='amount',
    BIN_HOURS=N_HOURS,       # each bin = 2 hr
    N_BIN=N_BINS,          # 3 bins = total 6 hr window
    AGG_FUN=('sum',), # total fluid input volume
    prefix='input',
    fill_missing=True
)
print(input_features_table)


    hadm_id    input_sum
0    103770     0.000000
1    105331     0.000000
2    110244     0.000000
3    111115     0.000000
4    124073     0.000000
5    126949     0.000000
6    132349     0.000000
7    140372     0.000000
8    142345     0.000000
9    145203     0.000000
10   157235     0.000000
11   157466     0.000000
12   165520     0.000000
13   167957     0.000000
14   172082  2718.857087
15   174739     0.000000
16   175237  1002.436809
17   177759     0.000000
18   182839     0.000000
19   187023     0.000000
20   188574     0.000000
21   189483     0.000000
22   199207     0.000000
23   199395     0.000000


In [29]:
# Finding output ids for

# OUTPUT
output_keywords = ['urine out']
outputs_ids = find_itemids_by_label(d_items, output_keywords, fuzzy=True)
print(outputs_ids)
# lactate_ids1 = [50813]



    itemid                        label abbreviation category
0    42042              ANGIO URINE OUT          NaN      NaN
1    46748           Cath Lab Urine Out          NaN      NaN
2    42666               E.R. URINE OUT          NaN      NaN
3    44237               E.R. urine out          NaN      NaN
4    45415                 ED Urine OUT          NaN      NaN
5    44103                 ER urine out          NaN      NaN
6    42892                 EW URINE OUT          NaN      NaN
7    42765             FARR 6 URINE OUT          NaN      NaN
8    43931              Floor urine out          NaN      NaN
9    44132          Procedure urine out          NaN      NaN
10   43053                    URINE OUT          NaN      NaN
11   46177              URINE OUT-ANGIO          NaN      NaN
12   40094        Urine Out Condom Cath          NaN      NaN
13   40055              Urine Out Foley          NaN      NaN
14   40473        Urine Out IleoConduit          NaN      NaN
15   400

In [33]:
# loading output values

# outputs = load_temporal_table(
#     filename='OUTPUTEVENTS.csv',
#     usecols=['hadm_id','charttime','itemid','valueuom'],
#     time_col='charttime',
#     value_col='value',
#     item_col='itemid',
#     itemids=outputs_ids
# )

# need to find sum across time bin
cols_out = ['hadm_id', 'itemid', 'charttime', 'value']
outputs = pd.read_csv(
    f"{datapath}/OUTPUTEVENTS.csv", usecols=cols_out, header=1,low_memory=False)

outputs_features_table = summarize_over_window(
    df=outputs,
    ref_times=ref_times,
    hadm_col='hadm_id',
    time_col='charttime',
    value_col='value',
    BIN_HOURS=N_HOURS,       # each bin = 2 hr
    N_BIN=N_BINS,          # 3 bins = total 6 hr window
    AGG_FUN=('sum',), # total fluid input volume
    prefix='outputs',
    fill_missing=True
)
print(outputs_features_table)




    hadm_id  outputs_sum
0    103770        595.0
1    105331       1715.0
2    110244       1670.0
3    111115        535.0
4    124073       1080.0
5    126949       2434.0
6    132349        390.0
7    140372       1930.0
8    142345         10.0
9    145203       1320.0
10   157235        250.0
11   157466       2552.0
12   165520        160.0
13   167957       2590.0
14   172082       2590.0
15   174739        920.0
16   175237         12.0
17   177759       5452.0
18   182839       3580.0
19   187023       1400.0
20   188574        765.0
21   189483        505.0
22   199207       2160.0
23   199395        780.0


In [37]:
# COMBINING ALL INTO SINGLE

static_features_table['hadm_id'] = pd.to_numeric(static_features_table['hadm_id'], errors='coerce').astype('Int64')
vitals_feature_table['hadm_id'] = pd.to_numeric(vitals_feature_table['hadm_id'], errors='coerce').astype('Int64')
labs_feature_table['hadm_id'] = pd.to_numeric(labs_feature_table['hadm_id'], errors='coerce').astype('Int64')
micro_features_table['hadm_id'] = pd.to_numeric(micro_features_table['hadm_id'], errors='coerce').astype('Int64')
input_features_table['hadm_id'] = pd.to_numeric(input_features_table['hadm_id'], errors='coerce').astype('Int64')
outputs_features_table['hadm_id'] = pd.to_numeric(outputs_features_table['hadm_id'], errors='coerce').astype('Int64')


all_features = [static_features_table, vitals_feature_table, labs_feature_table, micro_features_table, input_features_table, outputs_features_table]

all_features_table = reduce(
    lambda left, right: pd.merge(left, right, on='hadm_id', how='outer'),  all_features)

#print(all_features_table)
all_features_table.head()

#all_features_table.to_csv(f"{datapath}/all_features_table.csv", index=False)



/tmp/ipython-input-2852462716.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  static_features_table['hadm_id'] = pd.to_numeric(static_features_table['hadm_id'], errors='coerce').astype('Int64')


,hadm_id,gender,admission_type,ethnicity,pulse_bin1_mean,pulse_bin2_mean,pulse_bin3_mean,change_pulse_bin2_mean,change_pulse_bin3_mean,pulse_bin1_max,...,blood_pO2_bin3_mean,change_blood_pO2_bin2_mean,change_blood_pO2_bin3_mean,num_organisms,any_pos_culture,prop_resist,prop_intermed,num_antibiotics,input_sum,outputs_sum
0,103770,F,EMERGENCY,WHITE,65.428571,62.000000,63.000000,-3.428571,1.000000,71.0,...,NaN,NaN,NaN,2.0,1.0,0.078947,0.0,38.0,0.0,595.0
1,105331,F,EMERGENCY,UNKNOWN/NOT SPECIFIED,117.666667,136.833333,141.375000,19.166667,4.541667,136.0,...,103.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1715.0
2,110244,M,ELECTIVE,WHITE,66.166667,73.833333,76.666667,7.666667,2.833333,73.0,...,114.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1670.0
3,111115,F,EMERGENCY,WHITE,85.000000,86.666667,82.000000,1.666667,-4.666667,88.0,...,69.0,NaN,NaN,1.0,1.0,0.200000,0.0,5.0,0.0,535.0
4,124073,F,EMERGENCY,UNKNOWN/NOT SPECIFIED,64.571429,65.400000,69.142857,0.828571,3.742857,67.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1080.0


In [38]:
# Embedding the text columns - gender, admission type, ethnicity

import pandas as pd

df = all_features_table.copy()

# Columns to encode
categorical_cols = ["gender", "ethnicity", "admission_type"]

# Ensure they are hashable (strings)
for col in categorical_cols:
    df[col] = df[col].astype(str)

# One-hot encode
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=False)

df_encoded.head()
#print(df_encoded)

df_encoded.to_csv(f"{datapath}/all_features_table_encoded.csv", index=False)

# New Section